# Tutorial: Applying Saved Masks with `XarrayMask`

This notebook shows how to apply a **saved** geographic mask to model or analysis
output represented as an `xarray.Dataset` or `xarray.DataArray` — without using
`Cutout.add_mask` or `Cutout.mask`.

For contributor notes on the xarray masking design, see
[development/xarray_mask_workflow](../development/xarray_mask_workflow.rst).
To build masks from rasters and shapefiles, see
[mask creation workflow](mask_creation_workflow.ipynb).

## Overview

| Step | API | Module |
|------|-----|--------|
| Create and save a mask | `geodata.Mask` | `src/geodata/mask.py` |
| Run a model (wind, pvlib, …) | model `estimate()` | `src/geodata/model/` |
| Align mask to your grid, attach or apply | `geodata.XarrayMask` | `src/geodata/mask/xarray_mask.py` |

**`XarrayMask` does not replace mask creation.** It loads a saved mask and applies it
to xarray data on your target grid.

## Setup

This tutorial runs **offline** using a small synthetic grid and a temporary mask
directory. The same API calls work for production masks saved under `GEODATA_ROOT`.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import rasterio as ras
import shapely.geometry
import xarray as xr
from rasterio.transform import from_bounds

from geodata import Mask, XarrayMask

## Step 1: Stand in for model output

Your analysis dataset can use `x`/`y` or `lat`/`lon`. `XarrayMask` normalizes
coordinates via `ds_reformat_index` before alignment.

Below we use a small `(time, y, x)` dataset as if it came from a wind or PV model.

In [ ]:
y = np.array([30.75, 30.5, 30.25, 30.0])
x = np.array([100.0, 100.25, 100.5, 100.75])
time = np.array(["2016-01-01T00:00:00", "2016-01-01T01:00:00"], dtype="datetime64[ns]")

values = np.arange(len(time) * len(y) * len(x), dtype=np.float32).reshape(
    len(time), len(y), len(x)
)
model_ds = xr.Dataset(
    {"signal": (("time", "y", "x"), values)},
    coords={"time": time, "y": y, "x": x},
)
model_ds

## Step 2: Create and save a mask (offline example)

In practice you build masks with `Mask.add_layer`, `filter_layer`, `merge_layer`,
and `save_mask()` — see [mask creation workflow](mask_creation_workflow.ipynb).

Mask rasters are often stored at **higher resolution** than model output.
`XarrayMask` coarsens them onto `grid` automatically.

The helper below mirrors `tests/pr/mask/test_xarray_mask.py`.

In [ ]:
mask_dir = Path(tempfile.mkdtemp(prefix="geodata_xmask_tutorial_"))
mask_name = "tutorial_mask"

lon_step = float(np.abs(x[1] - x[0]))
lat_step = float(np.abs(y[1] - y[0]))
west = float(x.min() - lon_step / 2)
east = float(x.max() + lon_step / 2)
south = float(y.min() - lat_step / 2)
north = float(y.max() + lat_step / 2)

nlon_hi = len(x) * 2
nlat_hi = len(y) * 2
transform = from_bounds(west, south, east, north, nlon_hi, nlat_hi)

arr = np.zeros((nlat_hi, nlon_hi), dtype=np.uint8)
arr[nlat_hi // 4 : 3 * nlat_hi // 4, nlon_hi // 4 : 3 * nlat_hi // 4] = 1

layer_path = mask_dir / "source.tif"
with ras.open(
    str(layer_path),
    "w",
    driver="GTiff",
    height=arr.shape[0],
    width=arr.shape[1],
    count=1,
    dtype=arr.dtype,
    compress="lzw",
    crs="+proj=latlong",
    transform=transform,
) as dst:
    dst.write(arr, 1)

mask = Mask(name=mask_name, mask_dir=str(mask_dir))
mask.add_layer(str(layer_path), layer_name="source")
mask.merge_layer(show_raster=False)

region = shapely.geometry.box(west, south, (west + east) / 2, (south + north) / 2)
mask.extract_shapes({"region_a": region}, show_raster=False)
mask.save_mask()

print(f"Saved mask '{mask_name}' under {mask_dir}")

## Step 3: Load and align — `XarrayMask.from_name`

Pass your model grid so the saved mask is coarsened and aligned to the same
`x`/`y` (or `lat`/`lon`) coordinates.

In [ ]:
xmask = XarrayMask.from_name(mask_name, grid=model_ds, mask_dir=str(mask_dir))
xmask

You can also build from an in-memory `Mask` object:

```python
loaded = Mask.from_name(mask_name, mask_dir=str(mask_dir))
xmask = XarrayMask.from_mask(loaded, grid=model_ds)
```

## Step 4: Attach — legacy-compatible output

`attach()` returns a dict of datasets (keys: `merged_mask`, plus any shape masks).
Each dataset contains your original variables plus `mask` and optional `area` — the
same structure as `Cutout.mask()`.

In [ ]:
attached = xmask.attach(model_ds, include_area=True)
list(attached.keys())

In [ ]:
merged = attached["merged_mask"]
merged

## Step 5: Apply — filtered outputs

- `mode="where"` — set values outside the mask to NaN
- `mode="multiply"` — set values outside the mask to zero

In [ ]:
where_out = xmask.apply(model_ds, mode="where", include_area=True)["merged_mask"]
multiply_out = xmask.apply(model_ds, mode="multiply", include_area=False)["merged_mask"]

where_out["signal"].isel(time=0)

## Step 6: Area-weighted aggregation

With `attach(..., include_area=True)` you can compute mask- and area-weighted
statistics over time — the same pattern as the legacy Cutout workflow.

In [ ]:
ds = attached["merged_mask"]
weighted_mean = (
    (ds["signal"] * ds["mask"] * ds["area"]).sum(dim=["lat", "lon"])
    / (ds["mask"] * ds["area"]).sum(dim=["lat", "lon"])
)
weighted_mean

## Production usage

When your mask is already saved under the default mask directory (`GEODATA_ROOT`):

```python
xmask = XarrayMask.from_name("china", grid=output_ds)  # uses geodata.config.MASK_DIR
masked = xmask.apply(output_ds, mode="where")
```

### Typical pipeline

1. `output_ds = model.estimate(...)`
2. `xmask = XarrayMask.from_name("my_mask", grid=output_ds, mask_dir=...)`
3. `xmask.attach(output_ds)` or `xmask.apply(output_ds, ...)`

### See also

| Topic | Page |
|-------|------|
| Create masks from GIS layers | [mask_creation_workflow](mask_creation_workflow.ipynb) |
| Legacy Cutout masking | [mask_on_cutout](../legacy/mask_on_cutout.ipynb) |
| Xarray masking design notes (contributors) | [xarray_mask_workflow](../development/xarray_mask_workflow.rst) |
| Migration plan (contributors) | [mask_xarray_migration_plan](../development/mask_xarray_migration_plan.md) |
| Automated examples | `tests/pr/mask/test_xarray_mask.py`, `tests/pr/test_wind_xarraymask_integration.py` |